In [ ]:
import tkinter as tk  # biblioteca pro front end(GUI)
from tkinter import messagebox  # importa a subblioteca messagebox pra exibir alertas em pop-ups




# ESTRUTURA DE DADOS: ANDARES COMO LISTA DUPLAMENTE ENCADEADA

class node:  # classe dos nós que vai definir os andares
    def __init__(self, andar): 
        self.andar = andar  #número do andar atual
        self.proxE = None  #ponteiro para o próximo andar (andar de cima)
        self.backE = None  #ponteiro para o andar anterior (andar de baixo)

# ---------------------------------------------------------
# ESTRUTURA DE DADOS: FILA SIMPLESMENTE ENCADEADA (CHAMADAS)
# ---------------------------------------------------------
class nodeF:  # Define a classe do nó para a Fila de chamadas
    def __init__(self, valor):  # Construtor do nó da fila, recebe o valor (tupla com origem e destino)
        self.valor = valor  # Armazena os dados da chamada
        self.prox = None  # Ponteiro para a próxima chamada na fila

class filaEncadeada():  # Define a classe da Fila Encadeada (FIFO - First In, First Out)
    def __init__(self):  # Construtor da fila
        self.start = None  # Ponteiro para o início da fila
        self.fim = None  # Ponteiro para o final da fila

    def adicionar(self, atributo):  # Método para enfileirar uma nova chamada
        novo = nodeF(atributo)  # Cria um novo nó com a chamada
        if self.start is None and self.fim is None:  # Se a fila estiver vazia...
            self.start = novo  # O início da fila passa a ser o novo nó
            self.fim = novo  # O fim da fila também passa a ser o novo nó
        else:  # Se já houver elementos na fila...
            self.fim.prox = novo  # O antigo último nó aponta para o novo nó
            self.fim = novo  # Atualiza o ponteiro de 'fim' para ser o novo nó
    
    def remover(self):  # Método para desenfileirar a próxima chamada
        aux = self.start  # Salva o nó atual do início em uma variável auxiliar
        if self.start is None:  # Se a fila estiver vazia...
            return None  # Retorna 'None' (nada a remover)
        elif self.start is self.fim:  # Se houver apenas UM elemento na fila...
            self.start = None  # A fila fica vazia (início nulo)
            self.fim = None  # A fila fica vazia (fim nulo)
            return aux.valor  # Retorna o valor que foi removido
        else:  # Se houver mais de um elemento...
            self.start = self.start.prox  # O início da fila avança para o próximo nó
            return aux.valor  # Retorna o valor do nó que estava no início
            
    def esta_vazia(self):  # Método que verifica se a fila está vazia
        return self.start is None  # Retorna True se 'start' for None, senão False

# ---------------------------------------------------------
# LÓGICA DO ELEVADOR (MÁQUINA DE ESTADOS)
# ---------------------------------------------------------
class elevador:  # Classe que representa as ações e estado de um elevador
    def __init__(self, id_elevador):  # Construtor do elevador, recebe um ID numérico
        self.id_elevador = id_elevador  # Armazena o ID
        self.head = None  # Início da lista duplamente encadeada de andares (térreo)
        self.end = None   # Fim da lista de andares (último andar)
        self.atual = None  # Ponteiro para o andar onde o elevador está agora
        self.chamada_atual = None  # Armazena a tupla (origem, destino) da viagem atual
        
        self.estado = "PARADO"  # Define o estado inicial da máquina de estados
        self.ticks_espera = 0  # Contador para simular o tempo de embarque/desembarque

    def inserir(self, andar):  # Método para popular os andares (constrói a lista encadeada)
        novo_nodo = node(andar)  # Cria o nó do andar
        if self.head is None:  # Se for o primeiro andar inserido...
            self.head = novo_nodo  # Ele é o início
            self.end = novo_nodo  # Ele também é o fim
            self.atual = novo_nodo  # O elevador começa neste andar
        else:  # Se já houver andares...
            self.end.proxE = novo_nodo  # O último andar atual aponta (para cima) para o novo
            novo_nodo.backE = self.end  # O novo andar aponta (para baixo) para o antigo último
            self.end = novo_nodo  # Atualiza o fim da lista para o novo andar

    def processar_passo(self):  # Método responsável pela movimentação a cada atualização do sistema
        """Esta função é chamada várias vezes por segundo pelo Tkinter"""
        # Se o elevador está esperando (portas abrindo/fechando), apenas diminui o tempo
        if self.ticks_espera > 0:  # Se o temporizador de portas for maior que zero...
            self.ticks_espera -= 1  # Deduz 1 'tick' do tempo de espera
            return  # Encerra o turno deste elevador

        if self.chamada_atual is None:  # Se não há chamada designada para este elevador...
            self.estado = "PARADO"  # Ele entra/permanece no estado "PARADO"
            return  # Encerra o turno

        origem, destino = self.chamada_atual  # Desempacota a tupla da chamada em origem e destino

        # MÁQUINA DE ESTADOS: Lida com as diferentes fases da viagem
        if self.estado == "INDO_ORIGEM":  # Se o elevador está a caminho do passageiro...
            if self.atual.andar != origem:  # Se ainda não chegou no andar de origem...
                # Dá um "passo" na lista encadeada para subir ou descer
                if self.atual.andar < origem: self.atual = self.atual.proxE  # Sobe um andar
                else: self.atual = self.atual.backE  # Desce um andar
            else:  # Se chegou na origem...
                self.estado = "EMBARCANDO"  # Muda o estado para embarque
                self.ticks_espera = 3  # Espera 3 "ticks" para simular o embarque

        elif self.estado == "EMBARCANDO":  # Se o passageiro terminou de embarcar...
            self.estado = "INDO_DESTINO"  # Muda o estado para ir ao destino

        elif self.estado == "INDO_DESTINO":  # Se o elevador está com passageiro indo ao destino...
            if self.atual.andar != destino:  # Se ainda não chegou no destino...
                if self.atual.andar < destino: self.atual = self.atual.proxE  # Sobe
                else: self.atual = self.atual.backE  # Desce
            else:  # Se chegou no destino...
                self.estado = "DESEMBARCANDO"  # Muda o estado para o passageiro sair
                self.ticks_espera = 3  # Adiciona atraso de 3 'ticks' para portas/desembarque

        elif self.estado == "DESEMBARCANDO":  # Ao final do desembarque...
            self.estado = "PARADO"  # O elevador fica parado novamente
            self.chamada_atual = None # Fim da viagem, libera o elevador para novas chamadas

# ---------------------------------------------------------
# GERENCIADOR DO PRÉDIO (DESPACHANTE)
# ---------------------------------------------------------
class despachante:  # Classe que gerencia a frota de elevadores e a fila de chamadas
    def __init__(self, fila_global, total_andares=10):
        self.conj = []  # vetor que guarda cada elevador
        self.fila = fila_global  # Referência para a fila de chamadas pendentes
        self.total_andares = total_andares  #número de andares
        
    def adicionar_elevadores(self, quantidade=3):  # função que inicia os elevadores
        for i in range(quantidade):  #de acordo com a quantidade
            novo_elevador = elevador(i+1)  #cria um objeto elevador com id 1,2,3...
            for b in range(self.total_andares):  #loop pelos andares
                novo_elevador.inserir(b)  #constrói a lista duplamente encadeada de andares neste elevador
            self.conj.append(novo_elevador)  # Adiciona o elevador pronto à lista do prédio
            
    def gerenciar_predio(self):  # A função mestre chamada a cada 'tick' do simulador
        """Distribui as chamadas e faz os elevadores andarem 1 passo"""
        # 1. Tenta atribuir novas chamadas aos elevadores parados
        if not self.fila.esta_vazia():  # Se existe alguém esperando na fila...
            # Cria uma lista apenas com elevadores com estado "PARADO"
            disponiveis = [e for e in self.conj if e.estado == "PARADO"] 
            if disponiveis:  # Se há pelo menos um elevador disponível...
                chamada = self.fila.remover()  # Tira a chamada da fila de espera
                if chamada:  # Segurança: garante que retornou dados
                    origem, destino = chamada  # Desempacota
                    # Encontra o elevador disponível que está mais perto da origem (menor distância absoluta)
                    mais_prox = min(disponiveis, key=lambda e: abs(e.atual.andar - origem))
                    mais_prox.chamada_atual = (origem, destino)  # Atribui a corrida a ele
                    mais_prox.estado = "INDO_ORIGEM"  # Muda seu estado para começar a mover

        # 2. Manda cada elevador executar sua ação do momento
        for e in self.conj:  # Itera sobre todos os elevadores do prédio
            e.processar_passo()  # Executa a lógica de movimento/estado de cada um

# ---------------------------------------------------------
# FRONTEND (Tkinter UI)
# ---------------------------------------------------------
class AppElevadores:  # Classe responsável pela interface gráfica
    def __init__(self, root, despachante_predio):  # Construtor da janela
        self.root = root  # Referência da janela raiz do Tkinter
        self.predio = despachante_predio  # Referência do gerenciador lógico
        self.root.title("Simulador Event-Driven (Sem Threads)")  # Título da janela
        self.root.geometry("600x650")  # Tamanho da janela (Largura x Altura)

        # Adiciona chamadas iniciais pré-programadas à fila do prédio
        self.predio.fila.adicionar((1, 8))  # Viagem do andar 1 ao 8
        self.predio.fila.adicionar((2, 0))  # Viagem do andar 2 ao 0 (térreo)
        self.predio.fila.adicionar((6, 9))  # Viagem do andar 6 ao 9

        self.montar_interface()  # Chama a função que desenha os botões e campos
        
        # Inicia o "Game Loop" do Tkinter (o loop de atualização do sistema)
        self.loop_principal()

    def montar_interface(self):  # Constrói os elementos (widgets) na tela
        frame_top = tk.Frame(self.root)  # Cria um container na parte superior
        frame_top.pack(pady=10)  # Empacota na janela com um espaçamento vertical

        tk.Label(frame_top, text="Origem:").grid(row=0, column=0, padx=5)  # Texto 'Origem:'
        self.origem_entry = tk.Entry(frame_top, width=5)  # Campo de texto para digitar a origem
        self.origem_entry.grid(row=0, column=1)  # Posiciona o campo

        tk.Label(frame_top, text="Destino:").grid(row=0, column=2, padx=5)  # Texto 'Destino:'
        self.destino_entry = tk.Entry(frame_top, width=5)  # Campo de texto para o destino
        self.destino_entry.grid(row=0, column=3)  # Posiciona o campo

        # Botão que executa a função 'adicionar_chamada' ao ser clicado
        tk.Button(frame_top, text="Adicionar Chamada", command=self.adicionar_chamada, bg="lightblue").grid(row=0, column=4, padx=15)
        # Botão que fecha o programa ao ser clicado (destruindo a janela)
        tk.Button(frame_top, text="Sair", command=lambda:self.root.destroy(), bg="salmon").grid(row=0, column=5)

        self.canvas_h = 500  # Define a altura do Canvas (área de desenho)
        self.canvas_w = 500  # Define a largura do Canvas
        # Cria a área de desenho onde os elevadores vão aparecer
        self.canvas = tk.Canvas(self.root, width=self.canvas_w, height=self.canvas_h, bg="white", highlightthickness=1, highlightbackground="black")
        self.canvas.pack(pady=10)  # Empacota o Canvas na janela

    def adicionar_chamada(self):  # Função disparada pelo botão "Adicionar Chamada"
        try:
            o = int(self.origem_entry.get())  # Lê e converte para inteiro o valor da Origem
            d = int(self.destino_entry.get())  # Lê e converte para inteiro o valor do Destino
            # Verifica se os andares digitados existem no prédio
            if 0 <= o < self.predio.total_andares and 0 <= d < self.predio.total_andares:
                self.predio.fila.adicionar((o, d))  # Coloca a viagem na fila
                self.origem_entry.delete(0, tk.END)  # Limpa o campo de origem
                self.destino_entry.delete(0, tk.END)  # Limpa o campo de destino
            else:
                # Mostra aviso se os andares estiverem fora do intervalo permitido
                messagebox.showwarning("Erro", f"Os andares vão de 0 a {self.predio.total_andares - 1}.")
        except ValueError:  # Se o usuário não digitou um número...
            messagebox.showerror("Erro", "Digite inteiros válidos.")  # Mostra erro

    def loop_principal(self):  # Função cíclica que cria a animação e processamento em tempo real
        """O Coração do programa. Executa a lógica e desenha, repetidamente."""
        # 1. Roda a lógica do prédio (1 passo): distribui chamadas e move os elevadores
        self.predio.gerenciar_predio()

        # 2. Desenha o novo estado na tela
        self.canvas.delete("all")  # Apaga tudo que foi desenhado no ciclo anterior
        altura_andar = self.canvas_h / self.predio.total_andares  # Calcula o tamanho visual de cada andar
        
        for i in range(self.predio.total_andares):  # Desenha as linhas que representam os andares
            y = self.canvas_h - (i * altura_andar)  # Calcula a coordenada Y do chão do andar (inverte eixo Y)
            self.canvas.create_line(0, y, self.canvas_w, y, fill="lightgray", dash=(4, 2))  # Desenha a linha pontilhada
            self.canvas.create_text(25, y - altura_andar/2, text=f"{i}º", font=("Arial", 12, "bold"))  # Escreve o número do andar

        largura_elevador = 50  # Define a largura visual do retângulo do elevador
        for idx, el in enumerate(self.predio.conj):  # Itera pelos elevadores para desenhá-los
            if el.atual:  # Se o elevador tem um andar atual válido...
                andar_atual = el.atual.andar  # Pega o número do andar para o cálculo visual
                x0 = 100 + (idx * 120)  # Calcula a posição horizontal (eixo X) baseada no índice
                y1 = self.canvas_h - (andar_atual * altura_andar)  # Calcula a base do elevador
                y0 = y1 - altura_andar  # Calcula o topo do elevador

                # Cor verde se "PARADO", Laranja se em movimento/ocupado
                cor_fundo = "#4CAF50" if el.estado == "PARADO" else "#FF9800"
                # Desenha o retângulo que representa o elevador
                self.canvas.create_rectangle(x0, y0 + 5, x0 + largura_elevador, y1, fill=cor_fundo, outline="black")
                # Escreve "E1", "E2", etc., no meio do elevador
                self.canvas.create_text(x0 + largura_elevador/2, y0 + 20, text=f"E{idx+1}", fill="white", font=("Arial", 10, "bold"))
                
                # Exibe o estado textual da máquina de estados abaixo do elevador
                self.canvas.create_text(x0 + largura_elevador/2, y1 + 10, text=el.estado, fill="blue", font=("Arial", 8))

        # 3. Agenda o próximo ciclo para daqui a 1000 milissegundos (1 segundo)
        # O método 'after' chama self.loop_principal novamente, criando o loop do jogo
        self.root.after(1000, self.loop_principal)


# rodando


fila_chamadas = filaEncadeada()
    

predio = despachante(fila_chamadas, total_andares=10)  #inicia o prédio com 10 andares
predio.adicionar_elevadores(quantidade=3)  # 3 elevadores
    
root = tk.Tk()  # Inicia o motor principal da interface gráfica do Tkinter
app = AppElevadores(root, predio)  # Cria o aplicativo passando a janela e a lógica do prédio
root.mainloop()  # Bloqueia a execução aqui, mantendo a janela aberta e respondendo a eventos